# E12: chạy lại OTI trên ImageNet bằng câu GPT mới, chia 3 phần

Mỗi phần nghịch đảo khoảng 33.000 ảnh ImageNet (một shard cũ `oti_B32_s0/s1/s2`) bằng bộ câu `P3_qwen15i_allf`.

| | A100 | L4 |
|---|---|---|
| Một phần chạy riêng | khoảng 12 giờ | khoảng 24 giờ |
| Ba phần cùng lúc trên một A100 | khoảng 17,5 giờ cho cả ba | |

**Cách chạy**

1. Thời gian chạy → Thay đổi loại thời gian chạy → GPU A100 (hoặc L4).
2. Ô 1: đặt `PARTS` (ví dụ `[1]`, `[2, 3]` hoặc `[1, 2, 3]`). Dùng **tài khoản Google khác** thì đặt `DRIVE_DIR` tới lối tắt của thư mục SEARLE dùng chung (xem bên dưới).
3. Chạy ô 1 → 4 theo thứ tự. Ô 2 hỏi quyền Google Drive.
4. Để ô 4 chạy (giữ kết nối, in tiến độ mỗi 5 phút) và giữ tab mở.

**Ghi chú**

- Một phần chỉ được chạy ở **một** runtime tại một thời điểm (hai runtime cùng ghi một phần sẽ hỏng token).
- Bị ngắt giữa chừng: chạy lại đúng phần đó, nó làm tiếp từ chỗ dừng (token được lưu mỗi 5 batch).
- Phần xong cuối cùng tự gộp 3 phần, train lại φ (`phi_B32_P3oti`, khoảng 3 giờ trên A100), đánh giá có khoảng tin cậy và sinh file test `phiP3oti_cirr`, `phiP3oti_circo`.
- Tài khoản khác: tài khoản đó cần quyền **Người chỉnh sửa** trên thư mục SEARLE; trong Drive của nó, thêm lối tắt của thư mục SEARLE vào "Drive của tôi" và đổi tên lối tắt (ví dụ `SEARLE_main`) rồi đặt `DRIVE_DIR = '/content/drive/MyDrive/SEARLE_main'`.
- Log: `queue/q14_part<N>.log` trong thư mục SEARLE.


In [ ]:
# Ô 1: chọn phần, thư mục SEARLE và kiểm tra GPU
PARTS = [1]                                   # các phần chạy trong runtime này: [1], [2, 3], [1, 2, 3], ...
DRIVE_DIR = '/content/drive/MyDrive/SEARLE'   # tài khoản khác: '/content/drive/MyDrive/SEARLE_main' (lối tắt)
import subprocess
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip() or 'không có GPU')


In [ ]:
# Ô 2: gắn Drive và khởi động môi trường (code từ GitHub, venv, dữ liệu; khoảng 25 phút lần đầu)
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess
assert os.path.isdir(f'{DRIVE_DIR}/queue'), f'không thấy {DRIVE_DIR}/queue: kiểm tra DRIVE_DIR / lối tắt'
open(f'{DRIVE_DIR}/queue/.write_test', 'w').write('ok'); os.remove(f'{DRIVE_DIR}/queue/.write_test')  # cần quyền ghi
if not os.path.exists('/content/boot.log'):
    boot = ('git clone -q https://github.com/chilk1307/Chiss.git /content/XLA_SEARLE 2>/dev/null; '
            'git -C /content/XLA_SEARLE pull -q; '
            f'SEARLE_DRIVE={DRIVE_DIR} SEARLE_BOOT_QUEUES="" bash /content/XLA_SEARLE/SEARLE-main/colab/queue/boot.sh')
    subprocess.Popen(f'setsid nohup bash -c \'{boot}\' > /content/boot.log 2>&1 < /dev/null &', shell=True)
    print('boot started')
else:
    print('boot already started')


In [ ]:
# Ô 3: chờ khởi động xong rồi chạy các phần PARTS của q14 (chạy nền, dừng ô này không dừng q14)
import subprocess, time
while 'boot done' not in open('/content/boot.log', errors='replace').read():
    time.sleep(30)
for PART in PARTS:
    cmd = (f"source /content/env.sh; git -C /content/XLA_SEARLE fetch -q origin && git -C /content/XLA_SEARLE reset -q --hard origin/main; "
           f"cp $SRC/colab/queue/q14.sh $DRIVE/queue/q14.sh; "
           f"pgrep -f 'q14.sh {PART}$' > /dev/null && echo 'q14 part {PART} already running' || "
           f"(setsid nohup bash $DRIVE/queue/q14.sh {PART} >> $DRIVE/queue/q14_part{PART}.log 2>&1 < /dev/null & echo 'q14 part {PART} started')")
    print(subprocess.run(['bash', '-c', cmd], capture_output=True, text=True).stdout)


In [ ]:
# Ô 4: giữ kết nối + theo dõi các phần PARTS (in mỗi 5 phút; dừng ô này không dừng q14)
import subprocess, time
from IPython.display import clear_output
lines = [f"echo \"== part {P}: $(tail -n 1 $D/queue/q14_part{P}.log | cut -c1-120)\"; "
         f"tail -c 800 $D/results/E12.0-oti-s{P - 1}/stdout.log 2>/dev/null | tr '\\r' '\\n' | grep -a 'it\\]' | tail -n 1 | cut -c1-110; "
         for P in PARTS]
cmd = (f"D={DRIVE_DIR}; date; " + ''.join(lines) +
       "tail -c 2000 $D/results/E12.2-train-phi_B32_P3oti/stdout.log 2>/dev/null | tr '\\r' '\\n' | grep -a '/100' | tail -n 1 | cut -c1-120; "
       "nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader")
for i in range(36 * 12):
    out = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True, encoding='utf-8', errors='replace').stdout
    clear_output(wait=True); print(out, flush=True)
    time.sleep(300)
